In [1]:
!pip install -q langchain langchain-community langchain-openai chromadb pypdf tiktoken


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.5/2.5 MB 39.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.2/1.2 MB 34.1 MB/s eta 0:00:00


In [31]:
# Install the Google-specific LangChain library
!pip install -q -U langchain-google-genai

import os
from google.colab import userdata
from langchain_google_genai import ChatGoogleGenerativeAI, GoogleGenerativeAIEmbeddings

# Access your key from the Colab Secrets
os.environ["GOOGLE_API_KEY"] = userdata.get('GOOGLE_API_KEY')

# Initialize Gemini models
# We use 'gemini-flash-latest' for the assistant and 'gemini-embedding-001' for the vector DB
llm = ChatGoogleGenerativeAI(model="models/gemini-flash-latest")
embeddings = GoogleGenerativeAIEmbeddings(model="models/gemini-embedding-001")

print("Gemini Assistant is ready!")

Gemini Assistant is ready!


In [3]:
from langchain_community.document_loaders import PyPDFLoader

# Replace "your_file.pdf" with the actual name of the file you uploaded
loader = PyPDFLoader("your_data.pdf")
data = loader.load()

print(f"Successfully loaded {len(data)} pages.")

Successfully loaded 12 pages.


In [4]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

text_splitter = RecursiveCharacterTextSplitter(chunk_size=1000, chunk_overlap=200)
chunks = text_splitter.split_documents(data)

print(f"Your document is now split into {len(chunks)} smaller chunks.")

Your document is now split into 31 smaller chunks.


In [19]:
from langchain_community.vectorstores import Chroma
import google.generativeai as genai
import os
from google.colab import userdata

# This creates a local database in your Colab folder
vector_db = Chroma.from_documents(
    documents=chunks,
    embedding=embeddings, # This is the Gemini embedding model you set up earlier
    persist_directory="./chroma_db"
)

print("Vector database is ready and stocked!")

# Ensure API key is configured for listing models
if "GOOGLE_API_KEY" not in os.environ:
    os.environ["GOOGLE_API_KEY"] = userdata.get('GOOGLE_API_KEY')
genai.configure(api_key=os.environ["GOOGLE_API_KEY"])

print("\nAvailable Embedding Models (for debugging):")
found_embedding_model = False
for m in genai.list_models():
    if "embedContent" in m.supported_generation_methods:
        print(f"- {m.name}")
        found_embedding_model = True

if not found_embedding_model:
    print("No embedding models found supporting 'embedContent'. Please check your API key and permissions.")

Vector database is ready and stocked!

Available Embedding Models (for debugging):
- models/gemini-embedding-001
- models/gemini-embedding-2-preview
- models/gemini-embedding-2


In [6]:
!pip install -q langchain-classic

In [7]:
!pip install -q -U langchain-core langchain-google-genai langchain-community

In [15]:
import os
from google.colab import userdata

# Bypass the main 'chains' shortcut and go directly to the sub-libraries
try:
    from langchain.chains.combine_documents import create_stuff_documents_chain
    from langchain.chains.retrieval import create_retrieval_chain
    from langchain_core.prompts import ChatPromptTemplate
    from langchain_google_genai import ChatGoogleGenerativeAI, GoogleGenerativeAIEmbeddings
    print("✅ Success! The direct paths worked.")
except ImportError:
    # If that still fails, we use the 2026 'langchain_core' standard
    from langchain_core.runnables import RunnablePassthrough
    from langchain_core.output_parsers import StrOutputParser
    print("✅ Success! Switched to Core Runnables.")

✅ Success! Switched to Core Runnables.


In [32]:
import os
from google.colab import userdata

# 1. THE IMPORTS (Explicitly from the core libraries)
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.runnables import RunnablePassthrough
from langchain_core.output_parsers import StrOutputParser
from langchain_google_genai import ChatGoogleGenerativeAI, GoogleGenerativeAIEmbeddings

# 2. THE SETUP
# Make sure you have 'GOOGLE_API_KEY' in your Colab Secrets (Key icon)
os.environ["GOOGLE_API_KEY"] = userdata.get('GOOGLE_API_KEY')

llm = ChatGoogleGenerativeAI(model="models/gemini-flash-latest") # Changed model to models/gemini-flash-latest
embeddings = GoogleGenerativeAIEmbeddings(model="models/text-embedding-004")

# 3. THE PROMPT
template = """Answer the question based only on the following context:
{context}

Question: {question}
"""
prompt = ChatPromptTemplate.from_template(template)

# 4. THE CHAIN (LCEL Style - the most stable way in 2026)
def format_docs(docs):
    return "\n\n".join(doc.page_content for doc in docs)

# NOTE: This requires your 'vector_db' to be defined in a previous cell.
# If you get an error here, re-run your Chroma.from_documents(...) cell!
rag_chain = (
    {"context": vector_db.as_retriever() | format_docs, "question": RunnablePassthrough()}
    | prompt
    | llm
    | StrOutputParser()
)

print("✅ Assistant is ready! Run the next cell to ask a question.")

✅ Assistant is ready! Run the next cell to ask a question.


In [26]:
import google.generativeai as genai
import os
from google.colab import userdata

# Ensure API key is configured
if "GOOGLE_API_KEY" not in os.environ:
    os.environ["GOOGLE_API_KEY"] = userdata.get('GOOGLE_API_KEY')
genai.configure(api_key=os.environ["GOOGLE_API_KEY"])

print("\nAvailable Chat Models (supporting 'generateContent'):")
found_chat_model = False
for m in genai.list_models():
    if "generateContent" in m.supported_generation_methods:
        print(f"- {m.name}")
        found_chat_model = True

if not found_chat_model:
    print("No chat models found supporting 'generateContent'. Please check your API key and permissions.")



Available Chat Models (supporting 'generateContent'):
- models/gemini-2.5-flash
- models/gemini-2.5-pro
- models/gemini-2.0-flash
- models/gemini-2.0-flash-001
- models/gemini-2.0-flash-lite-001
- models/gemini-2.0-flash-lite
- models/gemini-2.5-flash-preview-tts
- models/gemini-2.5-pro-preview-tts
- models/gemma-3-1b-it
- models/gemma-3-4b-it
- models/gemma-3-12b-it
- models/gemma-3-27b-it
- models/gemma-3n-e4b-it
- models/gemma-3n-e2b-it
- models/gemma-4-26b-a4b-it
- models/gemma-4-31b-it
- models/gemini-flash-latest
- models/gemini-flash-lite-latest
- models/gemini-pro-latest
- models/gemini-2.5-flash-lite
- models/gemini-2.5-flash-image
- models/gemini-3-pro-preview
- models/gemini-3-flash-preview
- models/gemini-3.1-pro-preview
- models/gemini-3.1-pro-preview-customtools
- models/gemini-3.1-flash-lite-preview
- models/gemini-3-pro-image-preview
- models/nano-banana-pro-preview
- models/gemini-3.1-flash-image-preview
- models/lyria-3-clip-preview
- models/lyria-3-pro-preview
- mod

In [45]:
print("--- Customer Support Assistant (Type 'exit' to stop) ---")

while True:
    user_input = input("You: ")
    if user_input.lower() in ["exit", "quit", "stop"]:
        break

    # This invokes the chain we built in the previous step
    response = rag_chain.invoke(user_input)

    print(f"\nAssistant: {response}\n")
    print("-" * 30)

--- Customer Support Assistant (Type 'exit' to stop) ---
You: brand name ?

Assistant: Based on the context provided, the primary brand name is **Samsung**. 

Other brand names mentioned in the trademark information include:
*   HDMI
*   Linux
*   Microsoft
*   Windows
*   PowerPoint
*   OpenGL

------------------------------
You: what is the band name of the product?

Assistant: Based on the context provided, the brand name of the product is **Samsung**.

------------------------------
You: quit


In [36]:
!pip install -q -U langgraph

In [40]:
from langgraph.graph import StateGraph, END
from typing import TypedDict, Optional

# 1. Define the State
class AgentState(TypedDict):
    question: str
    answer: str
    is_relevant: bool

# 2. Node: The AI checks the manual
def check_manual(state: AgentState):
    # We ask the AI to answer OR say "I don't know"
    prompt = f"Answer this based ONLY on the manual: {state['question']}. If it's not in the manual, respond with exactly 'NOT_FOUND'."
    response = rag_chain.invoke(prompt)

    if "NOT_FOUND" in response:
        return {"answer": "", "is_relevant": False}
    else:
        return {"answer": response, "is_relevant": True}

# 3. Node: The Human (Programmer) Input
def programmer_input(state: AgentState):
    print(f"\n⚠️ The manual does not have info for: '{state['question']}'")
    manual_response = input("Programmer, please type the correct response for the user: ")
    return {"answer": manual_response, "is_relevant": True}

# 4. Routing Logic
def route_question(state: AgentState):
    # This is the decision maker
    if state["is_relevant"]:
        return "end"
    else:
        return "human"

# 5. Build the Graph
workflow = StateGraph(AgentState)

workflow.add_node("assistant", check_manual)
workflow.add_node("human_reviewer", programmer_input)

workflow.set_entry_point("assistant")

# Add a Conditional Edge
workflow.add_conditional_edges(
    "assistant",
    route_question,
    {
        "end": END,
        "human": "human_reviewer"
    }
)

workflow.add_edge("human_reviewer", END)

# Compile
app = workflow.compile()

In [49]:
user_q = input("Ask a question: ")
inputs = {"question": user_q}

# This runs the graph and prints the final state
final_state = app.invoke(inputs)
print("\n--- THE ANSWER ---")
print(final_state["answer"])

Ask a question: i want orange juice

⚠️ The manual does not have info for: 'i want orange juice'
Programmer, please type the correct response for the user: the query is not relevant .

--- THE ANSWER ---
the query is not relevant .


In [46]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive
